In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, data.target
n_qubits = 4

In [2]:
x = ParameterVector("x", n_qubits)
simple_circuit = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    simple_circuit.ry(x[i], i)

entangled_circuit = zz_feature_map(n_qubits, reps=2)

In [7]:
def run_once(seed):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    scaler = StandardScaler().fit(X_train)
    pca = PCA(n_components=n_qubits).fit(scaler.transform(X_train))
    to_angle = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(scaler.transform(X_train)))
    prepare = lambda A: to_angle.transform(pca.transform(scaler.transform(A)))
    X_train_q, X_test_q = prepare(X_train), prepare(X_test)

    models = {
        "Classical SVM": SVC(kernel="rbf"),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Quantum SVM (simple)": QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=simple_circuit)),
        "Quantum SVM (entangled)": QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=entangled_circuit)),
    }

    rows = []
    for name, model in models.items():
        model.fit(X_train_q, y_train)
        pred = model.predict(X_test_q)
        rows.append({
            "split": seed,
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "balanced accuracy": balanced_accuracy_score(y_test, pred),
            "sensitivity": recall_score(y_test, pred, pos_label=0),   # 0 = cancer
        })
    return rows

In [10]:
rows = []
for seed in range(10):
    rows += run_once(seed)

results = pd.DataFrame(rows)
summary = results.groupby("model")[["accuracy", "balanced accuracy", "sensitivity"]].agg(["mean", "std"]).round(3)
print(summary.to_string())

                        accuracy        balanced accuracy        sensitivity       
                            mean    std              mean    std        mean    std
model                                                                              
Classical SVM              0.961  0.019             0.959  0.021       0.948  0.031
Logistic Regression        0.967  0.016             0.960  0.020       0.933  0.037
Quantum SVM (entangled)    0.787  0.026             0.753  0.033       0.626  0.067
Quantum SVM (simple)       0.965  0.015             0.959  0.018       0.938  0.034


In [11]:
os.makedirs("../results", exist_ok=True)
results.to_csv("../results/exp1_breast_cancer_10_splits.csv", index=False)


In [12]:
from scipy.stats import wilcoxon

acc = results.pivot(index="split", columns="model", values="accuracy")

for other in ["Quantum SVM (simple)", "Quantum SVM (entangled)", "Logistic Regression"]:
    stat, p = wilcoxon(acc[other], acc["Classical SVM"], zero_method="zsplit")
    print(f"{other:25s} vs Classical SVM:  p = {p:.3f}")

Quantum SVM (simple)      vs Classical SVM:  p = 0.312
Quantum SVM (entangled)   vs Classical SVM:  p = 0.002
Logistic Regression       vs Classical SVM:  p = 0.219
